## Transformer Training Essentials

Transformer Training Essentials
Introduction

Welcome back to Bringing Transformers to Life: Training & Inference! You've made excellent progress in this course. In your first lesson, you assembled a complete Transformer architecture, integrating all the components we've built throughout this learning journey. In the second lesson, you created a robust data preparation pipeline that transforms raw text into training-ready tensors, complete with vocabularies, special tokens, and dynamic batching.

Now we are at a pivotal point, as in today's lesson we'll be discussing training the Transformer. This is where everything comes together as we implement the actual training process: you'll learn how Transformers learn through autoregressive modeling, where the model predicts the next token given all previous tokens. We'll explore teacher forcing, a key training technique that accelerates learning, and implement sophisticated optimization strategies, including learning rate scheduling with warmup. By the end of this lesson, you'll have a complete training pipeline that can effectively train your Transformer model on sequence-to-sequence tasks.
Understanding Autoregressive Training Objectives

The autoregressive training objective forms the foundation of how Transformers learn to generate sequences. Unlike traditional machine learning, where we might predict a single output, autoregressive models learn to predict each token in a sequence conditioned on all previous tokens. This creates a natural decomposition of the sequence generation probability.

Mathematically, for a target sequence y=(y1,y2,...,yT)y=(y1​,y2​,...,yT​), the autoregressive objective maximizes the likelihood: P(y∣x)=∏t=1TP(yt∣y<t,x)P(y∣x)=∏t=1T​P(yt​∣y<t​,x) where xx is the source sequence and y<ty<t​ represents all tokens before position tt. This factorization allows the model to learn complex dependencies while maintaining computational tractability. During training, we use cross-entropy loss to measure how well the model's predicted probability distribution matches the true next token at each position.
Teacher Forcing: Training vs Inference

Teacher forcing is the standard way to train autoregressive Transformers (both encoder-decoder and decoder-only models).
During training we feed the ground-truth tokens into the decoder (or into the masked language-model input), while asking the model to predict the next token for every position. Because the model always sees the correct previous context, gradients are better behaved and convergence is much faster than if it had to consume its own, still-noisy predictions.

The downside is the resulting train–inference mismatch, often called exposure bias:

    Training: context consists of perfect tokens from the data set.
    Inference: context consists of the model's own predictions, which may contain mistakes that propagate.

Although exposure bias is an unwanted side-effect, in practice teacher forcing is still preferred because (1) it makes optimization tractable, (2) it yields state-of-the-art performance when combined with techniques such as scheduled sampling, label smoothing, or beam search, and (3) large, diverse data sets help the model learn to recover from occasional errors.
Implementing the Trainer with Learning Rate Scheduling

Let's begin implementing our TransformerTrainer class, which encapsulates all the training logic, including the sophisticated learning rate scheduling:
Python
Copy to clipboard

class TransformerTrainer:
    def __init__(self, model, train_loader, lr=1e-3, warmup_steps=20):
        self.model = model
        self.train_loader = train_loader
        self.start_lr = lr
        self.optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding tokens
        self.warmup_steps = warmup_steps
        self.step_num = 0

This initialization sets up the essential training components. We use Adam optimizer with specific beta values (0.9, 0.98) that work well for Transformers, following established best practices. The CrossEntropyLoss with ignore_index=0 ensures that padding tokens don't contribute to the loss calculation, which is crucial for variable-length sequences. The warmup_steps parameter controls how long the learning rate increases before beginning to decay.
Learning Rate Scheduling and Warmup

Effective learning rate scheduling is crucial for stable Transformer training. The original Transformer paper introduced a specific warmup schedule that gradually increases the learning rate during early training steps, then decreases it proportionally. This approach prevents the large parameter updates that can destabilize training in the early stages.

The warmup schedule follows the formula:
lr=dmodel−0.5⋅min⁡(step−0.5,step⋅warmup_steps−1.5)
lr=dmodel−0.5​⋅min(step−0.5,step⋅warmup_steps−1.5)

where dmodeldmodel​ is the model dimension. This creates a learning rate that increases linearly during warmup, then decreases with the inverse square root of the step number. This schedule helps the model converge more reliably while maintaining training stability throughout the process.

Let's implement this learning rate scheduling logic:
Python
Copy to clipboard

    def get_lr_scale(self):
        """Learning rate schedule with warmup"""
        d_model = self.model.d_model
        step_num = self.step_num + 1
        return (d_model ** -0.5) * min(step_num ** -0.5, step_num * (self.warmup_steps ** -1.5))
    
    def update_lr(self):
        """Update learning rate based on schedule"""
        lr_scale = self.get_lr_scale()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = self.start_lr * lr_scale

The get_lr_scale method implements the mathematical formula we discussed earlier, computing the scaling factor based on the current step and warmup duration. During warmup, the learning rate increases linearly, then transitions to inverse square root decay. The update_lr method applies this scaling to the optimizer's parameter groups, ensuring the learning rate updates dynamically throughout training.
The Training Loop: Masking

The core training logic handles teacher forcing and proper masking for both padding and causal attention:
Python
Copy to clipboard

    def train_epoch(self):
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        num_batches = 0
        
        for batch in self.train_loader:
            self.step_num += 1
            self.update_lr()
            
            # Get batch data
            src = batch['src']
            tgt_input = batch['tgt']
            tgt_output = batch['tgt_output']
            
            # Create masks
            src_mask = self.model.create_padding_mask(src)
            tgt_causal_mask = self.model.create_causal_mask(tgt_input.size(1))
            tgt_padding_mask = self.model.create_padding_mask(tgt_input)
            # Combine masks: both must be True for attention to be allowed
            # Broadcasting will handle the shape differences
            tgt_mask = tgt_causal_mask & tgt_padding_mask

This section demonstrates the dual masking strategy essential for proper Transformer training. The source mask prevents attention to padding tokens, while the target mask combines causal masking (preventing future token access) with padding masking. The & operator ensures both conditions must be satisfied for attention to occur, maintaining the autoregressive property while handling variable-length sequences.
The Training Loop: Optimization

The optimization phase of the training loop implements the complete forward-backward pass with loss computation:
Python
Copy to clipboard

            # Forward pass with teacher forcing
            self.optimizer.zero_grad()
            output = self.model(src, tgt_input, src_mask, tgt_mask)
            
            # Compute loss
            loss = self.criterion(output.reshape(-1, output.size(-1)), tgt_output.reshape(-1))
            
            # Backward pass
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
            
            if num_batches % 10 == 0:
                avg_loss = total_loss / num_batches
                lr = self.optimizer.param_groups[0]['lr']
                print(f"Step {self.step_num}, Loss: {avg_loss:.4f}, LR: {lr:.6f}")
        
        return total_loss / num_batches

This loop showcases teacher forcing in action: tgt_input contains the ground truth tokens (with <SOS> prefix), while tgt_output contains the targets (with <EOS> suffix). The model learns to predict each token in tgt_output given the corresponding prefix in tgt_input. The loss reshaping flattens the sequence dimension, treating each position as an independent classification problem across the vocabulary.
Training in Action: From Theory to Practice

Now let's examine the complete training pipeline that brings everything together:
Python
Copy to clipboard

def train_transformer():
    """Train Transformer model on synthetic data"""
    print("Training Transformer Model...")
    
    # Create data
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=200)
    
    # Build vocabularies
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)
    
    print(f"Source vocab size: {src_vocab.size}")
    print(f"Target vocab size: {tgt_vocab.size}")
    
    # Create dataset and dataloader
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=15)
    train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    
    # Create model
    model = Transformer(
        src_vocab_size=src_vocab.size,
        tgt_vocab_size=tgt_vocab.size,
        d_model=64,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=256,
        dropout=0.1
    )
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

This pipeline demonstrates the complete integration of all components we've built. We create synthetic data, build vocabularies, initialize the dataset and dataloader using our custom collate_fn function, and instantiate a Transformer model with appropriate hyperparameters. The model size (237,718 parameters) is reasonable for our synthetic task while being large enough to demonstrate meaningful learning dynamics.

The final training execution brings everything together:
Python
Copy to clipboard

    # Create trainer and train
    trainer = TransformerTrainer(model, train_loader, lr=1e-3, warmup_steps=25)
    
    # Training loop
    num_epochs = 3
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        avg_loss = trainer.train_epoch()
        print(f"Average loss: {avg_loss:.4f}")
    
    return model, src_vocab, tgt_vocab

def main():
    model, src_vocab, tgt_vocab = train_transformer()
    print("Training completed successfully!")

When we execute this training pipeline, we observe the following output:
text
Copy to clipboard

Training Transformer Model...
Source vocab size: 22
Target vocab size: 22
Model parameters: 237,718

Epoch 1/3
Step 10, Loss: 3.9631, LR: 0.000011
Step 20, Loss: 3.8995, LR: 0.000021
Average loss: 3.8170

Epoch 2/3
Step 35, Loss: 3.2435, LR: 0.000021
Step 45, Loss: 3.1208, LR: 0.000018
Average loss: 3.0270

Epoch 3/3
Step 60, Loss: 2.7883, LR: 0.000016
Step 70, Loss: 2.6777, LR: 0.000015
Average loss: 2.6533
Training completed successfully!

The training output reveals several important patterns that demonstrate successful learning. The loss decreases consistently from 3.8170 in the first epoch to 2.6533 in the third epoch, indicating the model is learning the word reversal task. The learning rate schedule is working correctly, starting very low (0.000011) during warmup, reaching its peak around step 20-25, then gradually decreasing. This step-by-step monitoring shows that individual batch losses are decreasing within each epoch, and the learning rate adjustments follow the expected warmup-then-decay pattern.
Conclusion and Next Steps

You have successfully implemented a complete Transformer training pipeline that demonstrates the fundamental principles of autoregressive learning. Your implementation incorporates teacher forcing for efficient training, sophisticated learning rate scheduling with warmup, proper masking for both padding and causal attention, and robust optimization strategies. The training results show clear evidence of learning, with consistent loss reduction and proper learning rate dynamics.

This comprehensive training framework provides the foundation for tackling real-world sequence-to-sequence tasks, from machine translation to text summarization. In the upcoming practice exercises, you'll have the opportunity to experiment with different hyperparameters, explore various training strategies, and gain hands-on experience with the nuances of Transformer training that will make you proficient in bringing these powerful models to life. Then, in the next and final of this course we'll be discussing inference strategies such as greedy decoding and beam search. Keep learning!



## Building Your First Training Foundation

Now that you understand the autoregressive training objective and key components like cross-entropy loss and teacher forcing, it's time to build the foundation of our training system. You'll implement the initialization method for the `TransformerTrainer` class, setting up all the essential components needed for training.

---

### Your Task

Your job is to complete the `__init__` method in the provided code. You need to:

- Store the `model` and `train_loader` as instance attributes.
- Set up an Adam optimizer with the correct parameters (`lr`, `betas=(0.9, 0.98)`, `eps=1e-9`).
- Create a cross-entropy loss function that ignores padding tokens (index 0).
- Initialize tracking variables for learning rate scheduling.

The `main.py` file will create a trainer instance and test your implementation — you should see "Trainer created successfully!" when everything works correctly. This initialization is the foundation that makes all future training possible!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.optim as optim
import math

from transformer.model import Transformer

class TransformerTrainer:
    def __init__(self, model, train_loader, lr=1e-3, warmup_steps=20):
        # TODO: Initialize the trainer with model, data loader, optimizer, loss function, and other training parameters
        pass
```

---

### Solution

The TODO is the whole body of `__init__`:

```python
def __init__(self, model, train_loader, lr=1e-3, warmup_steps=20):
    self.model = model
    self.train_loader = train_loader
    self.start_lr = lr
    self.optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
    self.criterion = nn.CrossEntropyLoss(ignore_index=0)
    self.warmup_steps = warmup_steps
    self.step_num = 0
```

- `self.model` and `self.train_loader` are stored so later methods such as `train_epoch` can use them.
- `self.start_lr` keeps the base learning rate. The warmup schedule multiplies it by a scale at every step, so it must not be overwritten.
- `optim.Adam(..., betas=(0.9, 0.98), eps=1e-9)` uses the settings from the original Transformer paper. The lower second beta (0.98 instead of 0.999) makes the optimizer react faster to gradient changes.
- `nn.CrossEntropyLoss(ignore_index=0)` skips positions whose target is `<PAD>` (index 0), so padding does not affect the loss.
- `self.warmup_steps` and `self.step_num = 0` are the tracking variables for the learning rate schedule: the step counter starts at 0 and the warmup length is kept.
- Without the fix, `__init__` is only `pass`, so the trainer has no `optimizer`, `criterion` or `step_num`, and any later call such as `trainer.train_epoch()` would raise an `AttributeError`.

The starter code imports `Transformer` from CodeSignal's `transformer` package, but `__init__` never uses it, so the code cell below drops the import. `main.py` is not part of the starter, so the code cell adds a labelled quick check at the end. It builds a trainer around a tiny `nn.Linear` stand-in model and an empty loader, checks the stored settings and prints "Trainer created successfully!".

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# In the notebook, the model is passed in when you create the trainer,
# so the CodeSignal-only `from transformer.model import Transformer` is not needed here.

class TransformerTrainer:
    def __init__(self, model, train_loader, lr=1e-3, warmup_steps=20):
        # Store the model and data loader
        self.model = model
        self.train_loader = train_loader

        # Keep the base learning rate; the schedule scales it later
        self.start_lr = lr

        # Adam with the betas and eps recommended for Transformers
        self.optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)

        # Cross-entropy that ignores padding tokens (index 0)
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)

        # Tracking variables for learning rate scheduling
        self.warmup_steps = warmup_steps
        self.step_num = 0


# Quick check (not part of the starter code): main.py is not provided,
# so a tiny stand-in model and an empty loader are enough to build a trainer
def main():
    model = nn.Linear(4, 4)
    trainer = TransformerTrainer(model, train_loader=[], lr=1e-3, warmup_steps=20)

    assert trainer.model is model
    assert trainer.start_lr == 1e-3
    assert trainer.warmup_steps == 20 and trainer.step_num == 0
    assert trainer.optimizer.param_groups[0]['betas'] == (0.9, 0.98)
    assert trainer.optimizer.param_groups[0]['eps'] == 1e-9
    assert trainer.criterion.ignore_index == 0
    print("Trainer created successfully!")

if __name__ == "__main__":
    main()


## Building Smart Training Schedules

Now that you have completed the trainer setup, it's time to add the learning rate scheduling functionality. Learning rate scheduling is essential for transformer training — it uses a warmup period followed by gradual decay to help the model converge effectively.

---

### Your Task

Your job is to implement two key methods in the `TransformerTrainer` class:

- `get_lr_scale()` for calculating the learning rate scale factor using the warmup formula:

$$\text{lr} = d_{\text{model}}^{-0.5} \cdot \min\left(\text{step}^{-0.5},\ \text{step} \cdot \text{warmup\_steps}^{-1.5}\right)$$

- `update_lr()` to apply the scale factor to update all parameter groups in the optimizer.

The provided code includes helpful comments with the exact formula you need. With proper learning rate scheduling, your transformer will train much more stably and effectively!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.optim as optim
import math

from transformer.model import Transformer

class TransformerTrainer:
    def __init__(self, model, train_loader, lr=1e-3, warmup_steps=20):
        self.model = model
        self.train_loader = train_loader
        self.start_lr = lr
        self.optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding tokens
        self.warmup_steps = warmup_steps
        self.step_num = 0

    def get_lr_scale(self):
        """Learning rate schedule with warmup"""
        # TODO: Implement learning rate scaling with warmup
        pass

    def update_lr(self):
        """Update learning rate based on schedule"""
        # TODO: Get the learning rate scale and update all parameter groups
        # Use self.get_lr_scale() to get the scale factor
        # Apply it to self.start_lr and set it for all param_groups in the optimizer
        pass
```

---

### Solution

Two methods were missing:

```python
def get_lr_scale(self):
    """Learning rate schedule with warmup"""
    d_model = self.model.d_model
    step_num = self.step_num + 1
    return (d_model ** -0.5) * min(step_num ** -0.5, step_num * (self.warmup_steps ** -1.5))

def update_lr(self):
    """Update learning rate based on schedule"""
    lr_scale = self.get_lr_scale()
    for param_group in self.optimizer.param_groups:
        param_group['lr'] = self.start_lr * lr_scale
```

- `d_model ** -0.5` scales the whole schedule by the model size, so larger models get a smaller learning rate.
- `step_num = self.step_num + 1` shifts the counter by one. `self.step_num` starts at 0, and `0 ** -0.5` would raise a `ZeroDivisionError`.
- The `min` picks the active phase. While `step_num < warmup_steps`, `step_num * warmup_steps ** -1.5` is the smaller term, so the rate grows linearly. After the warmup, `step_num ** -0.5` is smaller, so the rate decays with the inverse square root. The two terms are equal exactly at `step_num == warmup_steps`, which is the peak.
- `update_lr` multiplies the base rate `self.start_lr` by the scale and writes it to every entry in `optimizer.param_groups`. This is how PyTorch lets you change the learning rate while training.
- Without the fix, both methods are only `pass`. `get_lr_scale()` returns `None` and `update_lr()` changes nothing, so the optimizer keeps the constant `lr` for the whole training and there is no warmup or decay.

The starter code imports `Transformer` from CodeSignal's `transformer` package, but the trainer never uses it, so the code cell below drops the import. `get_lr_scale` reads `self.model.d_model`, so the model you pass in needs a `d_model` attribute (the `Transformer` from the earlier lessons has one). The starter has no `main()`, so the code cell adds a labelled quick check at the end. It uses an `nn.Linear` stand-in with `d_model = 64` and prints the scale and learning rate at a few steps, which shows the warmup peak at step 20 and the decay after it.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# In the notebook, the model is passed in when you create the trainer,
# so the CodeSignal-only `from transformer.model import Transformer` is not needed here.

class TransformerTrainer:
    def __init__(self, model, train_loader, lr=1e-3, warmup_steps=20):
        self.model = model
        self.train_loader = train_loader
        self.start_lr = lr
        self.optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding tokens
        self.warmup_steps = warmup_steps
        self.step_num = 0

    def get_lr_scale(self):
        """Learning rate schedule with warmup"""
        d_model = self.model.d_model
        # step_num counts finished steps, so shift by one to avoid 0 ** -0.5
        step_num = self.step_num + 1
        # Linear warmup, then inverse square root decay
        return (d_model ** -0.5) * min(step_num ** -0.5, step_num * (self.warmup_steps ** -1.5))

    def update_lr(self):
        """Update learning rate based on schedule"""
        # Get the learning rate scale and apply it to the base learning rate
        lr_scale = self.get_lr_scale()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = self.start_lr * lr_scale


# Quick check (not part of the starter code): a stand-in model with d_model=64
# is enough to see the warmup and the decay
def main():
    model = nn.Linear(4, 4)
    model.d_model = 64
    trainer = TransformerTrainer(model, train_loader=[], lr=1e-3, warmup_steps=20)

    for step in [1, 10, 20, 40, 80]:
        trainer.step_num = step - 1  # get_lr_scale adds 1 itself
        trainer.update_lr()
        lr = trainer.optimizer.param_groups[0]['lr']
        print(f"Step {step:>2}: scale={trainer.get_lr_scale():.6f}, lr={lr:.6f}")

if __name__ == "__main__":
    main()


## Building the Training Engine

Now that you've set up learning rate scheduling, it's time to implement the core training logic. You'll complete the `train_epoch` method in the `TransformerTrainer` class, which handles the actual training loop for one full pass through the data.

---

### Your Task

The provided code includes detailed guidance in the comments. Your implementation should handle several key aspects:

- Set the model to training mode and initialize tracking variables.
- Loop through batches, updating the learning rate at each step.
- Extract batch data and create the necessary attention masks.
- Perform a forward pass with teacher forcing and compute the loss.
- Execute a backward pass and update model parameters.
- Track and display training progress.

This method brings together all the concepts from autoregressive training, such as teacher forcing, masked attention, and proper loss computation with padding token handling. Once you complete this, you'll have a fully functional training epoch that can learn to generate sequences step by step!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.optim as optim
import math

from transformer.model import Transformer

class TransformerTrainer:
    def __init__(self, model, train_loader, lr=1e-3, warmup_steps=20):
        self.model = model
        self.train_loader = train_loader
        self.start_lr = lr
        self.optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding tokens
        self.warmup_steps = warmup_steps
        self.step_num = 0

    def get_lr_scale(self):
        """Learning rate schedule with warmup"""
        d_model = self.model.d_model
        step_num = self.step_num + 1
        return (d_model ** -0.5) * min(step_num ** -0.5, step_num * (self.warmup_steps ** -1.5))

    def update_lr(self):
        """Update learning rate based on schedule"""
        lr_scale = self.get_lr_scale()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = self.start_lr * lr_scale

    def train_epoch(self):
        """Train for one epoch"""
        # TODO: Implement the training loop for one epoch
        # 1. Set model to training mode
        # 2. Initialize total_loss and num_batches counters
        # 3. Loop through batches in train_loader:
        #    - Update step_num and learning rate
        #    - Get batch data (src, tgt_input, tgt_output)
        #    - Create masks (src_mask, tgt_causal_mask, tgt_padding_mask)
        #    - Combine target masks (tgt_mask = tgt_causal_mask & tgt_padding_mask)
        #    - Forward pass with teacher forcing
        #    - Compute loss using criterion
        #    - Backward pass and optimizer step
        #    - Update loss tracking
        #    - Print progress every 10 batches
        # 4. Return average loss
        pass
```

---

### Solution

The TODO is the whole body of `train_epoch`:

```python
def train_epoch(self):
    """Train for one epoch"""
    self.model.train()
    total_loss = 0
    num_batches = 0

    for batch in self.train_loader:
        self.step_num += 1
        self.update_lr()

        src = batch['src']
        tgt_input = batch['tgt']
        tgt_output = batch['tgt_output']

        src_mask = self.model.create_padding_mask(src)
        tgt_causal_mask = self.model.create_causal_mask(tgt_input.size(1))
        tgt_padding_mask = self.model.create_padding_mask(tgt_input)
        tgt_mask = tgt_causal_mask & tgt_padding_mask

        self.optimizer.zero_grad()
        output = self.model(src, tgt_input, src_mask, tgt_mask)
        loss = self.criterion(output.reshape(-1, output.size(-1)), tgt_output.reshape(-1))

        loss.backward()
        self.optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        if num_batches % 10 == 0:
            avg_loss = total_loss / num_batches
            lr = self.optimizer.param_groups[0]['lr']
            print(f"Step {self.step_num}, Loss: {avg_loss:.4f}, LR: {lr:.6f}")

    return total_loss / num_batches
```

- `self.model.train()` turns on training behaviour such as dropout. `total_loss` and `num_batches` are the counters for the average.
- `self.step_num += 1` and `self.update_lr()` run once per batch, so the warmup schedule follows the number of steps, not epochs. `step_num` is not reset between epochs.
- The batch holds the teacher-forcing pair: `tgt` (starts with `<SOS>`) goes into the decoder, and `tgt_output` (ends with `<EOS>`) is what the model should predict.
- `src_mask` hides padding in the source. The target mask is the causal mask `&` the padding mask: a position can be attended to only if it is both in the past and not padding. Broadcasting handles the different shapes.
- `self.optimizer.zero_grad()` clears the gradients of the previous batch before the forward pass. Without it, gradients would accumulate across batches.
- The loss reshapes `output` from `(batch, seq_len, vocab)` to `(batch * seq_len, vocab)` and `tgt_output` to `(batch * seq_len,)`, so every position is one classification. `ignore_index=0` removes the padding positions.
- `loss.backward()` computes the gradients and `self.optimizer.step()` applies them.
- Without the fix, `train_epoch` is only `pass`. It returns `None`, no step is taken and the model never learns.

The starter code imports `Transformer` from CodeSignal's `transformer` package, but the trainer never uses it, so the code cell below drops the import. `train_epoch` relies on the model having `d_model`, `create_padding_mask`, `create_causal_mask` and a `forward(src, tgt, src_mask, tgt_mask)`, which the `Transformer` from the earlier lessons has. The starter has no `main()`, so the code cell adds a labelled quick check at the end. It uses a small `TinyModel` stand-in with those pieces, runs one epoch over 12 random batches and prints the average loss. The data is random and the model is only seeded with `torch.manual_seed(0)`, so the loss is not meaningful, it only shows that the loop runs.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# In the notebook, the model is passed in when you create the trainer,
# so the CodeSignal-only `from transformer.model import Transformer` is not needed here.

class TransformerTrainer:
    def __init__(self, model, train_loader, lr=1e-3, warmup_steps=20):
        self.model = model
        self.train_loader = train_loader
        self.start_lr = lr
        self.optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding tokens
        self.warmup_steps = warmup_steps
        self.step_num = 0

    def get_lr_scale(self):
        """Learning rate schedule with warmup"""
        d_model = self.model.d_model
        step_num = self.step_num + 1
        return (d_model ** -0.5) * min(step_num ** -0.5, step_num * (self.warmup_steps ** -1.5))

    def update_lr(self):
        """Update learning rate based on schedule"""
        lr_scale = self.get_lr_scale()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = self.start_lr * lr_scale

    def train_epoch(self):
        """Train for one epoch"""
        # Set model to training mode
        self.model.train()

        # Initialize loss tracking counters
        total_loss = 0
        num_batches = 0

        for batch in self.train_loader:
            # Update step_num and the learning rate (warmup schedule)
            self.step_num += 1
            self.update_lr()

            # Get batch data
            src = batch['src']
            tgt_input = batch['tgt']
            tgt_output = batch['tgt_output']

            # Create masks
            src_mask = self.model.create_padding_mask(src)
            tgt_causal_mask = self.model.create_causal_mask(tgt_input.size(1))
            tgt_padding_mask = self.model.create_padding_mask(tgt_input)
            # Combine masks: both must be True for attention to be allowed
            # Broadcasting will handle the shape differences
            tgt_mask = tgt_causal_mask & tgt_padding_mask

            # Forward pass with teacher forcing
            self.optimizer.zero_grad()
            output = self.model(src, tgt_input, src_mask, tgt_mask)

            # Compute loss: flatten (batch, seq_len, vocab) to (batch * seq_len, vocab)
            loss = self.criterion(output.reshape(-1, output.size(-1)), tgt_output.reshape(-1))

            # Backward pass and optimizer step
            loss.backward()
            self.optimizer.step()

            # Update loss tracking
            total_loss += loss.item()
            num_batches += 1

            # Print progress every 10 batches
            if num_batches % 10 == 0:
                avg_loss = total_loss / num_batches
                lr = self.optimizer.param_groups[0]['lr']
                print(f"Step {self.step_num}, Loss: {avg_loss:.4f}, LR: {lr:.6f}")

        # Return average loss
        return total_loss / num_batches


# Quick check (not part of the starter code): the stand-in model below only has
# what train_epoch needs: d_model, the two mask helpers and a forward pass that
# returns logits of shape (batch, seq_len, vocab)
class TinyModel(nn.Module):
    def __init__(self, vocab_size=10, d_model=16):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.proj = nn.Linear(d_model, vocab_size)

    def create_padding_mask(self, seq, pad_idx=0):
        return (seq != pad_idx).unsqueeze(1).unsqueeze(2)

    def create_causal_mask(self, size):
        return torch.tril(torch.ones(size, size)).bool()

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        return self.proj(self.embedding(tgt) + self.embedding(src).mean(dim=1, keepdim=True))

def main():
    torch.manual_seed(0)
    model = TinyModel()
    batches = [{'src': torch.randint(1, 10, (4, 5)),
                'tgt': torch.randint(1, 10, (4, 6)),
                'tgt_output': torch.randint(1, 10, (4, 6))} for _ in range(12)]
    trainer = TransformerTrainer(model, batches, lr=1e-3, warmup_steps=5)

    avg_loss = trainer.train_epoch()
    print(f"Average loss: {avg_loss:.4f}")
    print(f"Steps taken: {trainer.step_num}")

    assert trainer.step_num == len(batches)
    assert model.training

if __name__ == "__main__":
    main()


## Complete Your Training Script

Now that you have the `train_epoch` method working, it's time to complete the training script by implementing the final missing pieces. You need to finish two key components that will allow your transformer to train properly on batches of data.

---

### Your Task

First, implement the `collate_fn` function, which handles batching sequences of different lengths. This function should:

- Extract the source, target, and target output sequences from each batch.
- Use PyTorch's padding utilities to make all sequences the same length.
- Return a dictionary with the properly formatted batch data.

Second, complete the main training loop in the `train_transformer` function. You'll need to create a loop that runs for 3 epochs, calls the trainer's `train_epoch` method, and prints the progress and loss information.

Once you complete these functions, you'll have a fully working transformer training pipeline that can learn from your synthetic translation data!

---

### Starter Code

```python
import torch
from torch.utils.data import DataLoader

from transformer.model import Transformer
from data_utils import create_synthetic_data, Vocabulary, TranslationDataset
from train import TransformerTrainer

def collate_fn(batch):
    """Custom collate function"""
    # TODO: Extract src, tgt, and tgt_output from batch
    # TODO: Use torch.nn.utils.rnn.pad_sequence to pad sequences with padding_value=0
    # TODO: Return dictionary with keys: 'src', 'tgt', 'tgt_output'
    pass

def train_transformer():
    """Train Transformer model on synthetic data"""
    print("Training Transformer Model...")

    # Create data
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=200)

    # Build vocabularies
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)

    print(f"Source vocab size: {src_vocab.size}")
    print(f"Target vocab size: {tgt_vocab.size}")

    # Create dataset and dataloader
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=15)
    train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

    # Create model
    model = Transformer(
        src_vocab_size=src_vocab.size,
        tgt_vocab_size=tgt_vocab.size,
        d_model=64,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=256,
        dropout=0.1
    )

    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Create trainer and train
    trainer = TransformerTrainer(model, train_loader, lr=1e-3, warmup_steps=25)

    # TODO: Create training loop for 2 epochs
    # TODO: For each epoch, print epoch number and call trainer.train_epoch()
    # TODO: Print the average loss returned by train_epoch()

    return model, src_vocab, tgt_vocab

def main():
    model, src_vocab, tgt_vocab = train_transformer()
    print("Training completed successfully!")

if __name__ == "__main__":
    main()
```

---

### Solution

Two parts were missing. In `collate_fn`:

```python
def collate_fn(batch):
    """Custom collate function"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]

    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = torch.nn.utils.rnn.pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)

    return {
        'src': src_batch,
        'tgt': tgt_batch,
        'tgt_output': tgt_output_batch
    }
```

In `train_transformer`, the training loop:

```python
num_epochs = 3
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    avg_loss = trainer.train_epoch()
    print(f"Average loss: {avg_loss:.4f}")
```

- `collate_fn` gets a list of dataset items (dictionaries of 1-D tensors with different lengths). It collects each key into a list, and `pad_sequence(..., batch_first=True, padding_value=0)` stacks each list into one `(batch, seq_len)` tensor, padded with the `<PAD>` index 0. This is what `DataLoader` needs to build a batch.
- The loop calls `trainer.train_epoch()` once per epoch. It trains over the whole `train_loader` and returns the average loss, which is printed. `step_num` is kept inside the trainer, so the learning rate schedule continues across epochs.
- The task text asks for 3 epochs, but the TODO comment in the starter says 2. The solution uses 3, as in the task text and the lesson's example output.
- Without the fix, `collate_fn` returns `None`, so the first batch from the `DataLoader` fails and nothing can train. Even with `collate_fn` fixed, a missing loop means no epoch runs and the model is returned untrained.

The starter code imports `Transformer`, `Vocabulary`, `TranslationDataset`, `create_synthetic_data` and `TransformerTrainer` from CodeSignal modules (`transformer.model`, `data_utils`, `train`) that do not exist in this notebook. The code cell below defines all of them inline, using the versions built in the earlier exercises and lessons, so it is long but runs on its own. The data is shuffled and the model is not seeded, so the loss values differ from run to run.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
from collections import Counter
from torch.utils.data import Dataset, DataLoader

# The `transformer` package is CodeSignal-only, so the building blocks are defined inline.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """Multi-Head Attention module"""
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self._init_weights()

    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)

    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        output = torch.matmul(attention_weights, v)
        return output, attention_weights

    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)

        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)

        # Adjust mask for multi-head attention
        if mask is not None:
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]

        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)

        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)

        return output, attention_weights


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """Position-wise Feed-Forward Network"""
        super(PositionwiseFeedForward, self).__init__()

        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

        # Choose activation function
        if activation.lower() == 'relu':
            self.activation = F.relu
        elif activation.lower() == 'gelu':
            self.activation = F.gelu
        else:
            raise ValueError(f"Unsupported activation: {activation}")

        self._init_weights()

    def _init_weights(self):
        """Initialize weights"""
        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.xavier_uniform_(self.linear2.weight)
        nn.init.constant_(self.linear1.bias, 0)
        nn.init.constant_(self.linear2.bias, 0)

    def forward(self, x):
        """Forward pass through position-wise FFN"""
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x


class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        """Add & Norm layer: residual connection + layer normalization"""
        super(AddNorm, self).__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer_output):
        """Apply residual connection and layer normalization"""
        output = x + self.dropout(sublayer_output)
        return self.layer_norm(output)


class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """Token Embedding layer"""
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        nn.init.xavier_uniform_(self.embedding.weight)

    def forward(self, x):
        """Convert token indices to embeddings scaled by sqrt(d_model)"""
        return self.embedding(x) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        """Sinusoidal Positional Encoding"""
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)

        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # div_term will have shape (ceil(d_model / 2),)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                           (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)

        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """Add positional encoding to input embeddings"""
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)


class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Encoder Layer"""
        super(TransformerEncoderLayer, self).__init__()

        # Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)

        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)

        # Add & Norm layers
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)

    def forward(self, x, mask=None):
        """Forward pass through encoder layer"""
        # Multi-Head Self-Attention + Add & Norm
        attn_output, attention_weights = self.self_attention(x, x, x, mask)
        x = self.add_norm1(x, attn_output)

        # Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm2(x, ff_output)

        return x, attention_weights

class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Encoder Layers"""
        super(TransformerEncoder, self).__init__()

        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])

        self.num_layers = num_layers

    def forward(self, x, mask=None):
        """Forward pass through encoder stack"""
        attention_weights = []

        for layer in self.layers:
            x, attn_weights = layer(x, mask)
            attention_weights.append(attn_weights)

        return x, attention_weights


class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Decoder Layer"""
        super(TransformerDecoderLayer, self).__init__()

        # Masked Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)

        # Encoder-Decoder Multi-Head Attention
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)

        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)

        # Add & Norm layers: one per sub-layer
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)

    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder layer"""
        # 1. Masked Multi-Head Self-Attention + Add & Norm
        self_attn_output, self_attn_weights = self.self_attention(x, x, x, self_attention_mask)  # x is query, key and value
        x = self.add_norm1(x, self_attn_output)  # residual + layer norm

        # 2. Encoder-Decoder Multi-Head Attention + Add & Norm
        cross_attn_output, cross_attn_weights = self.cross_attention(
            x, encoder_output, encoder_output, cross_attention_mask)  # Q from x, K and V from the encoder
        x = self.add_norm2(x, cross_attn_output)  # residual + layer norm

        # 3. Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm3(x, ff_output)  # residual + layer norm

        return x, (self_attn_weights, cross_attn_weights)

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Decoder Layers"""
        super(TransformerDecoder, self).__init__()

        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])

        self.num_layers = num_layers

    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder stack"""
        # Initialize empty lists for storing attention weights from each layer
        all_self_attention_weights = []
        all_cross_attention_weights = []

        # Pass the inputs through each decoder layer in sequence
        for layer in self.layers:
            x, (self_attn_weights, cross_attn_weights) = layer(
                x, encoder_output, self_attention_mask, cross_attention_mask)  # x is updated by each layer
            all_self_attention_weights.append(self_attn_weights)  # collect per-layer weights
            all_cross_attention_weights.append(cross_attn_weights)

        # Return the final output and all collected attention weights
        return x, (all_self_attention_weights, all_cross_attention_weights)


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8,
                 num_encoder_layers=6, num_decoder_layers=6, d_ff=2048,
                 max_seq_len=5000, dropout=0.1, activation='relu'):
        """Complete Transformer model for sequence-to-sequence tasks"""
        super(Transformer, self).__init__()

        self.d_model = d_model

        # Embeddings
        self.src_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_embedding = TokenEmbedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len, dropout)

        # Encoder and Decoder
        self.encoder = TransformerEncoder(num_encoder_layers, d_model, num_heads, d_ff, dropout, activation)
        self.decoder = TransformerDecoder(num_decoder_layers, d_model, num_heads, d_ff, dropout, activation)

        # Output projection
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)

        self._init_parameters()

    def _init_parameters(self):
        """Initialize model parameters"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def create_padding_mask(self, seq, pad_token=0):
        """Create padding mask"""
        return (seq != pad_token).unsqueeze(1).unsqueeze(2)

    def create_causal_mask(self, size):
        """Create causal mask for decoder"""
        mask = torch.tril(torch.ones(size, size))
        return mask.unsqueeze(0).unsqueeze(1).bool()

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """Forward pass through complete Transformer"""
        # 1. Embed and encode the source sequence
        src_embedded = self.positional_encoding(self.src_embedding(src))
        encoder_output, _ = self.encoder(src_embedded, src_mask)

        # 2. Embed and decode the target sequence (cross-attention uses src_mask)
        tgt_embedded = self.positional_encoding(self.tgt_embedding(tgt))
        decoder_output, _ = self.decoder(tgt_embedded, encoder_output, tgt_mask, src_mask)

        # 3. Project to vocabulary size: (B, T, d_model) -> (B, T, tgt_vocab_size)
        return self.output_projection(decoder_output)


class Vocabulary:
    def __init__(self):
        self.token2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2token = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.size = 4

    def add_token(self, token):
        if token not in self.token2idx:
            self.token2idx[token] = self.size
            self.idx2token[self.size] = token
            self.size += 1

    def build_vocab(self, sentences, min_freq=1):
        """Build vocabulary from list of sentences"""
        counter = Counter()
        for sentence in sentences:
            counter.update(sentence.split())

        for token, freq in counter.items():
            if freq >= min_freq:
                self.add_token(token)

    def encode(self, sentence):
        """Convert sentence to token indices"""
        tokens = sentence.split()
        return [self.token2idx.get(token, self.token2idx['<UNK>'])
                for token in tokens]

    def decode(self, indices):
        """Convert token indices back to sentence"""
        tokens = [self.idx2token[idx] for idx in indices if idx != self.token2idx['<PAD>']]
        return ' '.join(tokens).replace('<SOS>', '').replace('<EOS>', '').strip()

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_sentence = self.src_sentences[idx]
        tgt_sentence = self.tgt_sentences[idx]

        # Encode sentences
        src_indices = self.src_vocab.encode(src_sentence)
        tgt_indices = self.tgt_vocab.encode('<SOS> ' + tgt_sentence)
        tgt_output = self.tgt_vocab.encode(tgt_sentence + ' <EOS>')

        return {
            'src': torch.tensor(src_indices, dtype=torch.long),
            'tgt': torch.tensor(tgt_indices, dtype=torch.long),
            'tgt_output': torch.tensor(tgt_output, dtype=torch.long)
        }

def create_synthetic_data(num_samples=1000):
    """Create synthetic translation data (English to reversed English)"""
    templates = [
        "hello world", "good morning", "how are you", "nice to meet you",
        "thank you very much", "see you later", "have a nice day"
    ]

    src_sentences = []
    tgt_sentences = []

    for _ in range(num_samples):
        # Pick a random template; the target is the same words in reverse order
        template = random.choice(templates)
        src_sentences.append(template)
        tgt_sentences.append(" ".join(template.split()[::-1]))

    return src_sentences, tgt_sentences


class TransformerTrainer:
    def __init__(self, model, train_loader, lr=1e-3, warmup_steps=20):
        self.model = model
        self.train_loader = train_loader
        self.start_lr = lr
        self.optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding tokens
        self.warmup_steps = warmup_steps
        self.step_num = 0

    def get_lr_scale(self):
        """Learning rate schedule with warmup"""
        d_model = self.model.d_model
        step_num = self.step_num + 1
        return (d_model ** -0.5) * min(step_num ** -0.5, step_num * (self.warmup_steps ** -1.5))

    def update_lr(self):
        """Update learning rate based on schedule"""
        lr_scale = self.get_lr_scale()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = self.start_lr * lr_scale

    def train_epoch(self):
        """Train for one epoch"""
        # Set model to training mode
        self.model.train()

        # Initialize loss tracking counters
        total_loss = 0
        num_batches = 0

        for batch in self.train_loader:
            # Update step_num and the learning rate (warmup schedule)
            self.step_num += 1
            self.update_lr()

            # Get batch data
            src = batch['src']
            tgt_input = batch['tgt']
            tgt_output = batch['tgt_output']

            # Create masks
            src_mask = self.model.create_padding_mask(src)
            tgt_causal_mask = self.model.create_causal_mask(tgt_input.size(1))
            tgt_padding_mask = self.model.create_padding_mask(tgt_input)
            # Combine masks: both must be True for attention to be allowed
            # Broadcasting will handle the shape differences
            tgt_mask = tgt_causal_mask & tgt_padding_mask

            # Forward pass with teacher forcing
            self.optimizer.zero_grad()
            output = self.model(src, tgt_input, src_mask, tgt_mask)

            # Compute loss: flatten (batch, seq_len, vocab) to (batch * seq_len, vocab)
            loss = self.criterion(output.reshape(-1, output.size(-1)), tgt_output.reshape(-1))

            # Backward pass and optimizer step
            loss.backward()
            self.optimizer.step()

            # Update loss tracking
            total_loss += loss.item()
            num_batches += 1

            # Print progress every 10 batches
            if num_batches % 10 == 0:
                avg_loss = total_loss / num_batches
                lr = self.optimizer.param_groups[0]['lr']
                print(f"Step {self.step_num}, Loss: {avg_loss:.4f}, LR: {lr:.6f}")

        # Return average loss
        return total_loss / num_batches


# In the notebook, Transformer, Vocabulary, TranslationDataset, create_synthetic_data
# and TransformerTrainer are defined inline below instead of being imported
# (CodeSignal had: from transformer.model import Transformer, from data_utils import ...,
# from train import TransformerTrainer)

def collate_fn(batch):
    """Custom collate function"""
    # Extract src, tgt, and tgt_output from batch
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]

    # Pad sequences to the longest one in the batch; 0 is the <PAD> index
    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = torch.nn.utils.rnn.pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)

    # Return dictionary with keys: 'src', 'tgt', 'tgt_output'
    return {
        'src': src_batch,
        'tgt': tgt_batch,
        'tgt_output': tgt_output_batch
    }

def train_transformer():
    """Train Transformer model on synthetic data"""
    print("Training Transformer Model...")

    # Create data
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=200)

    # Build vocabularies
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)

    print(f"Source vocab size: {src_vocab.size}")
    print(f"Target vocab size: {tgt_vocab.size}")

    # Create dataset and dataloader
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=15)
    train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

    # Create model
    model = Transformer(
        src_vocab_size=src_vocab.size,
        tgt_vocab_size=tgt_vocab.size,
        d_model=64,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=256,
        dropout=0.1
    )

    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Create trainer and train
    trainer = TransformerTrainer(model, train_loader, lr=1e-3, warmup_steps=25)

    # Training loop for 3 epochs (the task text says 3; the starter's comment says 2)
    num_epochs = 3
    for epoch in range(num_epochs):
        # Print the epoch number and train for one epoch
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        avg_loss = trainer.train_epoch()

        # Print the average loss returned by train_epoch()
        print(f"Average loss: {avg_loss:.4f}")

    return model, src_vocab, tgt_vocab

def main():
    model, src_vocab, tgt_vocab = train_transformer()
    print("Training completed successfully!")

if __name__ == "__main__":
    main()
